In [ ]:
# Parameters cell — Papermill overrides these at runtime
dataset_size = 10
start_base = 2440000
size_increment = 1000
contig = '2RL'
sample_set = "1354-VO-KE-DONNELLY-VMF00281"

In [ ]:
import malariagen_data

af1 = malariagen_data.Af1()

In [ ]:
import csv
import time
from pathlib import Path
import filelock
from memory_profiler import memory_usage
from collections.abc import Callable


def measure_regions(start_base: int, size_increment:int, dataset_size:int, sample_set:str,
                    contig: str, output_csv_prefix:str="memory_benchmarks"):
    sample_id = str(
        af1.sample_metadata(sample_sets=sample_set).iloc[0]["sample_id"]
    )


    output_files = {
        "snp_calls": Path(f"{output_csv_prefix}_snp_calls.csv"),
        "igv_views": Path(f"{output_csv_prefix}_igv_views.csv"),
    }

    def measure_and_write(output_path: Path, region:str, operation: Callable):
        start_time = time.perf_counter()
        peak_memory_mib = memory_usage((operation, ()), max_usage=True)
        runtime_sec = time.perf_counter() - start_time

        record = {
            "dataset_size": dataset_size,
            "size_increment": size_increment,
            "start_base": start_base,
            "sample_set": sample_set,
            "sample_id": sample_id,
            "region": region,
            "peak_memory_mib": round(peak_memory_mib, 2),
            "runtime_sec": round(runtime_sec, 4),
        }

        lock = filelock.FileLock(f"{output_path}.lock")
        with lock:
            file_exists = output_path.exists() and output_path.stat().st_size > 0
            with output_path.open("a", newline="") as f:
                writer = csv.DictWriter(f, fieldnames=record.keys())
                if not file_exists:
                    writer.writeheader()
                writer.writerow(record)

        print(f"Recorded {region} to {output_path}: {record}")

    for i in range(dataset_size):
        region_start = start_base + i * size_increment
        region = f"{contig}:{region_start}-{region_start + size_increment}"
        vcf_path = Path(
            "local_igv_vcfs",
            f"{sample_id}_{region}.vcf.gz",
        )
        vcf_path.parent.mkdir(parents=True, exist_ok=True)

        def create_vcf():
            af1.snp_calls_to_vcf(
                output_path=str(vcf_path),
                region=region,
                sample_sets=[sample_set],
                sample_query=f"sample_id == '{sample_id}'",
                overwrite=True,
            )

        measure_and_write(output_files["snp_calls"], region, create_vcf)

        def view_region():
            af1.view_alignments(
                sample=sample_id,
                region=region,
                snp_vcf_path=str(vcf_path),
            )

        measure_and_write(output_files["igv_views"], region, view_region)

In [2]:
measure_regions(
    start_base=start_base,
    size_increment=size_increment,
    dataset_size=dataset_size,
    sample_set=sample_set,
    contig=contig,
    output_csv_prefix=output_csv_prefix,
)
